In [ ]:
!pip install jax "jax[cuda13]" transformers huggingface_hub

# Qwen 3

**Paper:** [arXiv:2505.09388](https://arxiv.org/abs/2505.09388) — *Qwen3 Technical Report* (Qwen Team, Alibaba, 2025).  
**HF source:** [Qwen/Qwen3-30B-A3B](https://huggingface.co/Qwen/Qwen3-30B-A3B)

This notebook is a **from-scratch JAX implementation** of Qwen3-30B-A3B — a 30B parameter Mixture-of-Experts language model that activates only ~3B parameters per token. We run it across 4 GPUs using JAX's built-in device sharding.

---

### Model Overview

Qwen3-30B-A3B is a **sparse MoE transformer** — each token is routed to only 8 of 128 experts per layer, keeping compute close to a 3B dense model while drawing from a 30B parameter pool.

| Component | Value |
|---|---|
| Total parameters | ~30B |
| Active parameters / token | ~3B |
| Layers | 48 |
| Hidden size | 2,048 |
| Attention heads | 32 Q / 4 KV (GQA) |
| Head dim | 128 |
| Experts per layer | 128 |
| Active experts / token | 8 |
| Vocab size | 151,936 |

### Architecture at a glance

```
Input tokens  (seq_len,)
  ↓  Embedding lookup  →  (seq_len, 2048)
  ↓  48× Decoder Block:
       RMSNorm → GQA with RoPE + KV cache
       RMSNorm → MoE Router → top-8 of 128 SwiGLU experts
  ↓  Final RMSNorm
  ↓  LM Head  →  logits  (seq_len, 151936)
```

### Environment Setup

Disable Triton GEMM in XLA — avoids a numerical precision issue with MoE gate computations on some GPU driver versions.

In [ ]:
import os
os.environ['XLA_FLAGS'] = '--xla_gpu_enable_triton_gemm=false'

### Imports

In [ ]:
import glob
import json
import os
from dataclasses import dataclass
from functools import partial
from typing import Optional

import numpy as np
import jax
import jax.numpy as jnp
from jax.sharding import Mesh, NamedSharding, PartitionSpec as P
from safetensors.numpy import load_file
from transformers import AutoTokenizer

### Check Available Devices

Verify JAX can see all 4 GPUs before proceeding — the sharding setup below assumes a 4-device configuration.

In [ ]:
print(f"Devices available: {jax.devices()}") 

### Paths & Settings

Set the HuggingFace model name and local download path. Flip `USE_DUMMY_WEIGHTS = True` to skip the download and test the forward pass logic with random tensors.

In [ ]:
HF_MODEL_NAME = "Qwen/Qwen3-30B-A3B"
HF_MODEL_PATH = "Qwen3-30B-A3B"
USE_DUMMY_WEIGHTS = False

### HuggingFace Token

Qwen3 requires accepting the model licence on HF Hub. Paste your token here — it's only used for the one-time checkpoint download.

In [ ]:
os.environ["HF_TOKEN"] = "hf_XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX"  # YOUR HF TOKEN HERE

### Download Model Weights

Downloads all `.safetensors` shards from HF Hub to the local path (~60 GB, one-time). Skip this cell if you already have the weights on disk.

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download(repo_id=HF_MODEL_NAME, local_dir=HF_MODEL_PATH, local_dir_use_symlinks=False)

### Model Configuration

All architectural constants live in a frozen `Qwen3MoEConfig` dataclass. Marking it `frozen=True` lets JAX treat it as a compile-time static value when passed as a `static_argnames` to `jax.jit`.

In [ ]:
@dataclass(frozen=True)
class Qwen3MoEConfig:
    vocab_size: int
    hidden_size: int
    intermediate_size: int
    moe_intermediate_size: int
    num_hidden_layers: int
    num_attention_heads: int
    num_key_value_heads: int
    num_experts: int
    num_experts_per_tok: int
    head_dim: int
    rms_norm_eps: float
    rope_theta: float
    max_seq_len: int
    norm_topk_prob: bool

config = Qwen3MoEConfig(
    vocab_size=151936,
    hidden_size=2048,
    num_hidden_layers=48,
    num_attention_heads=32,
    num_key_value_heads=4,
    head_dim=128,
    intermediate_size=6144,
    moe_intermediate_size=768, 
    num_experts=128,       
    num_experts_per_tok=8,
    rms_norm_eps=1e-6,
    rope_theta=1000000.0,
    max_seq_len=512,
    norm_topk_prob=True
)

### Load Model Weights

The checkpoint is split across multiple `.safetensors` shards. We glob them all, load sequentially, and merge into a single `hf_weights` dict ready for extraction.

In [ ]:
if not USE_DUMMY_WEIGHTS:
    shard_paths = sorted(glob.glob(os.path.join(HF_MODEL_PATH, "*.safetensors")))
    
    if not shard_paths:
        raise FileNotFoundError(f"No safetensors found in {HF_MODEL_PATH}")

    combined_weights = {}
    for path in shard_paths:
        print(f"Loading shard: {os.path.basename(path)}...")
        shard = load_file(path)
        combined_weights.update(shard)
            
    hf_weights = combined_weights

### Device Mesh & Sharding Strategy

We arrange 4 GPUs in a `(4, 1)` mesh and name the single axis `model`. Each weight type gets a dedicated `NamedSharding` that tells JAX how to distribute it across devices:

| Sharding | Partition spec | Used for |
|---|---|---|
| `expert_sharding` | `P('model', None)` | Expert matrices — each GPU holds a subset of experts |
| `attention_sharding` | `P(None, 'model')` | Q/O projections — split along the output dimension |
| `replicate_sharding` | `P(None, None)` | K/V projections — replicated (only 4 KV heads, small) |
| `kv_cache_sharding` | `P(None, 'model', None)` | KV cache — split along the head dimension |
| `vocab_sharding` | `P('model', None)` | Embedding + LM head — sharded vocab |

In [ ]:
NUM_GPU = 4

devices = np.array(jax.devices()).reshape(NUM_GPU, 1)
mesh = Mesh(devices, axis_names=('model', 'data'))

expert_sharding = NamedSharding(mesh, P('model', None))
attention_sharding = NamedSharding(mesh, P(None, 'model'))
replicate_sharding = NamedSharding(mesh, P(None, None))
replicate_1d_sharding = NamedSharding(mesh, P(None))
vocab_sharding = NamedSharding(mesh, P('model', None))
kv_cache_sharding = NamedSharding(mesh, P(None, 'model', None))

### Weight Extraction

`get_w` pops a tensor from `hf_weights`, converts it to `bfloat16`, and optionally transposes and places it on the target devices.

`get_layer` builds one full decoder layer dict — stacking all 128 expert weight matrices into a single `(128, in, out)` array so we can index into them efficiently at runtime. This is the main weight extraction loop: expect it to take a few minutes.

In [ ]:
def get_w(
    name: str,
    transpose: bool = False,
    sharding_type: Optional[str] = None,
):
    val = hf_weights.pop(name)
    val = val.T if transpose else val

    sharding_map = {
        'expert': expert_sharding,
        'attention': attention_sharding,
        'replicate': replicate_sharding,
        'replicate_1d': replicate_1d_sharding,
        'vocab_parallel': vocab_sharding,
    }

    arr = jnp.array(val, dtype=jnp.bfloat16)
    if sharding_type in sharding_map:
        return jax.device_put(arr, sharding_map[sharding_type])
    return arr

def get_layer(
    i: int,
):
    stacked_gate_proj = jnp.stack([
        get_w(f'model.layers.{i}.mlp.experts.{j}.gate_proj.weight', True, sharding_type='expert') 
        for j in range(config.num_experts)
    ])
    
    stacked_up_proj = jnp.stack([
        get_w(f'model.layers.{i}.mlp.experts.{j}.up_proj.weight', True, sharding_type='expert') 
        for j in range(config.num_experts)
    ])
    
    stacked_down_proj = jnp.stack([
        get_w(f'model.layers.{i}.mlp.experts.{j}.down_proj.weight', True, sharding_type='expert') 
        for j in range(config.num_experts)
    ])
    
    layer = {
        'input_norm': {
            'scale': get_w(f'model.layers.{i}.input_layernorm.weight')
        },
        'mlp': {
            'experts': {
                'gate_proj': stacked_gate_proj,
                'up_proj':   stacked_up_proj,
                'down_proj': stacked_down_proj,
            },
            'gate': get_w(f'model.layers.{i}.mlp.gate.weight', True),
        },
        'attn': {
            'q_proj': get_w(f'model.layers.{i}.self_attn.q_proj.weight', True, sharding_type='attention'),
            'k_proj': get_w(f'model.layers.{i}.self_attn.k_proj.weight', True, sharding_type='replicate'),
            'v_proj': get_w(f'model.layers.{i}.self_attn.v_proj.weight', True, sharding_type='replicate'),
            'o_proj': get_w(f'model.layers.{i}.self_attn.o_proj.weight', True, sharding_type='attention'),
            'q_norm': {'scale': get_w(f'model.layers.{i}.self_attn.q_norm.weight', sharding_type='replicate_1d')},
            'k_norm': {'scale': get_w(f'model.layers.{i}.self_attn.k_norm.weight', sharding_type='replicate_1d')},
        },
        'post_attn_norm': {
            'scale': get_w(f'model.layers.{i}.post_attention_layernorm.weight')
        },
    }
    return layer

if not USE_DUMMY_WEIGHTS:
    layers = []
    for i in range(config.num_hidden_layers):
        layer = get_layer(i)
        layers.append(layer) 
        print("Completed Layer: ", i)

    m = {
        'lm_head': get_w('lm_head.weight', True, sharding_type='vocab_parallel'),
        'embed': get_w('model.embed_tokens.weight', sharding_type='vocab_parallel'),
        'layers': layers,
        'final_norm': get_w('model.norm.weight'),
    }


### Verify Weight Sharding

A quick visual check that the LM head is correctly split across all 4 devices. Each coloured slice represents one device's shard.

In [ ]:
jax.debug.visualize_array_sharding(m['lm_head'])

### KV Cache Allocation

Pre-allocate a zeroed KV cache for every layer — shape `(max_seq_len, num_kv_heads, head_dim)`. We fill it in-place during the forward pass using `dynamic_update_slice`. Sharding it along the head dimension keeps memory balanced across GPUs.

In [ ]:
def get_kv_cache(
    shape: tuple,
    sharding: Optional[NamedSharding] = None,
):
    cache = {
        'k': jnp.zeros(shape, dtype=jnp.bfloat16),
        'v': jnp.zeros(shape, dtype=jnp.bfloat16),
    }
    if sharding is not None:
        cache['k'] = jax.device_put(cache['k'], sharding)
        cache['v'] = jax.device_put(cache['v'], sharding)
    return cache


### Rotary Positional Embeddings (RoPE)

RoPE encodes position by rotating query and key vectors in 2D subspaces using precomputed sin/cos values. We build the full table once at startup (all positions up to `max_seq_len`) and slice into it per forward pass — no extra computation during inference.

$$\theta_i = \frac{1}{\text{rope\_theta}^{2i / d}}$$

Qwen3 uses a large `rope_theta = 1,000,000` which extends effective context length compared to the original `10,000` used in LLaMA.

In [ ]:
def qwen_rotational_embeddings(
    seq_len: int,
    head_dim: int,
    theta: float = 1000000.0,
):
    position_ids = jnp.arange(seq_len, dtype=jnp.bfloat16)
    freqs = 1.0 / (theta ** (jnp.arange(0, head_dim, 2, dtype=jnp.bfloat16) / head_dim))
    angles = position_ids[:, None] * freqs[None, :]
    
    cos = jnp.cos(angles)
    sin = jnp.sin(angles)
    
    return cos, sin

rotary_embeddings_table = qwen_rotational_embeddings(config.max_seq_len, config.head_dim, theta=config.rope_theta)
print("Cos Table shape: ", rotary_embeddings_table[0].shape, "Sin Table shape: ", rotary_embeddings_table[1].shape)


### RMS Normalisation

Qwen3 uses **RMSNorm** instead of standard LayerNorm — it skips the mean subtraction and normalises purely by root-mean-square. Simpler, faster, and works just as well in practice.

$$\text{RMSNorm}(x) = \frac{x}{\sqrt{\frac{1}{d}\sum x_i^2 + \varepsilon}} \cdot \gamma$$

In [ ]:
def rms_norm(
    x: jnp.ndarray,    # (seq_len, hidden_size)
    scale: jnp.ndarray, # (hidden_size,)
    eps: float = 1e-6,
):
    rms = jnp.mean(x ** 2, axis=-1, keepdims=True)
    x = x * jax.lax.rsqrt(rms + eps)
    return x * scale


### Applying RoPE

Three small helpers that wire together the positional encoding:

- **`rope_lookup`** — slices the precomputed sin/cos table to the current position range
- **`rotate_half`** — rearranges the last dim: `[x₁, x₂]` → `[-x₂, x₁]`, enabling the rotation
- **`apply_rope`** — applies the rotation in-place: `x·cos + rotate_half(x)·sin`

In [ ]:
def rope_lookup(
    position_ids: jnp.ndarray, # (seq_len,)
    rope_table: tuple,
):
    cos_table, sin_table = rope_table
    
    cos = cos_table[position_ids]
    sin = sin_table[position_ids]
    
    return cos, sin

def rotate_half(
    x: jnp.ndarray, # (..., head_dim)
):
    half = x.shape[-1] // 2
    x1 = x[..., :half]
    x2 = x[..., half:]
    return jnp.concatenate([-x2, x1], axis=-1)

def apply_rope(
    x: jnp.ndarray,   # (seq_len, num_heads, head_dim)
    cos: jnp.ndarray, # (seq_len, head_dim // 2)
    sin: jnp.ndarray, # (seq_len, head_dim // 2)
):
    cos = jnp.concatenate([cos, cos], axis=-1)
    sin = jnp.concatenate([sin, sin], axis=-1)
    
    cos = cos[:, None, :]
    sin = sin[:, None, :]

    return x * cos + rotate_half(x) * sin


### KV Cache Update

Writes the current step's K and V into the pre-allocated cache at `start_pos` using `jax.lax.dynamic_update_slice` — XLA's way to perform an in-place slice write inside a JIT-compiled function without triggering a copy.

In [ ]:
def update_cache(
    layer_cache: dict,
    k: jnp.ndarray, # (seq_len, num_kv_heads, head_dim)
    v: jnp.ndarray, # (seq_len, num_kv_heads, head_dim)
    start_pos: int,
):
    k = jax.lax.with_sharding_constraint(k, kv_cache_sharding)
    v = jax.lax.with_sharding_constraint(v, kv_cache_sharding)
    
    k_cache = jax.lax.dynamic_update_slice(layer_cache['k'], k, (start_pos, 0, 0))
    v_cache = jax.lax.dynamic_update_slice(layer_cache['v'], v, (start_pos, 0, 0))
    return k_cache, v_cache


### Causal Attention Mask

Builds a lower-triangular mask that prevents each query token from seeing future key positions. Values are `0.0` (allowed) or `-1e4` (blocked) — added directly to attention logits before softmax so blocked positions get near-zero weight.

In [ ]:
from functools import partial

@partial(jax.jit, static_argnums=(0,2))
def get_causal_mask(
    seq_len: int,
    start_pos: int,
    max_seq_len: int,
):
    q_idx = (jnp.arange(seq_len) + start_pos)[:, None]
    k_idx = jnp.arange(max_seq_len)[None, :]

    mask = (q_idx >= k_idx)
    mask = jnp.where(mask, 0.0, -1e4).astype(jnp.bfloat16)
    mask = mask[None, :, :]

    return mask


### Grouped-Query Attention (GQA)

Qwen3 uses **GQA** with 32 query heads but only 4 KV heads — an 8× reduction in KV cache memory vs full multi-head attention. Each KV head is shared across 8 query heads (`jnp.repeat` expands them before the matmul). Qwen3 also applies per-head **QK-Norm** (RMSNorm on Q and K separately) for training stability.

```
Q, K, V  =  linear projections of input
Q, K     →  per-head RMSNorm  →  RoPE
K, V     →  write into KV cache
logits   =  Q · Kᵀ / √head_dim  +  causal_mask
output   =  softmax(logits) · V  →  o_proj
```

In [ ]:
def qwen3_attention(
    input_embeddings: jnp.ndarray, # (seq_len, hidden_size)
    params: dict,
    config: Qwen3MoEConfig,
    position_embeddings: tuple,
    start_pos: int,
    layer_cache: dict,
):
    seq_len = input_embeddings.shape[0]
    
    q = jnp.dot(input_embeddings, params['attn']['q_proj'])
    k = jnp.dot(input_embeddings, params['attn']['k_proj'])
    v = jnp.dot(input_embeddings, params['attn']['v_proj'])

    q = q.reshape(seq_len, config.num_attention_heads, config.head_dim)
    k = k.reshape(seq_len, config.num_key_value_heads, config.head_dim)
    v = v.reshape(seq_len, config.num_key_value_heads, config.head_dim)

    q = rms_norm(q, params['attn']['q_norm']['scale'])
    k = rms_norm(k, params['attn']['k_norm']['scale'])

    cos, sin = position_embeddings
    q = apply_rope(q, cos, sin)
    k = apply_rope(k, cos, sin)

    k, v = update_cache(layer_cache, k, v, start_pos)
    updated_cache = {'k': k, 'v': v}

    # k = jnp.repeat(k, config.num_attention_heads // config.num_key_value_heads, axis=1)
    # v = jnp.repeat(v, config.num_attention_heads // config.num_key_value_heads, axis=1)

    q, k, v = q.transpose(1, 0, 2), k.transpose(1, 0, 2), v.transpose(1, 0, 2)

    logits = jnp.matmul(q, k.transpose(0, 2, 1)) / jnp.sqrt(config.head_dim)
    causal_mask = get_causal_mask(seq_len, start_pos, config.max_seq_len)

    logits = logits + causal_mask
    weights = jax.nn.softmax(logits, axis=-1)

    attn_output = jnp.matmul(weights, v)
    attn_output = attn_output.transpose(1, 0, 2).reshape(seq_len, -1)

    output = jnp.dot(attn_output, params['attn']['o_proj'])
    
    return output, updated_cache


### MoE Router

A single linear layer maps each token's hidden state to a probability distribution over all 128 experts. We select the top-8 by probability, then renormalise them to sum to 1 (`norm_topk_prob=True`) before using them as expert output weights.

In [ ]:
def qwen3_router(
    hidden_states: jnp.ndarray, # (seq_len, hidden_size)
    params: dict,
    config: Qwen3MoEConfig,
):
    router_logits = jnp.dot(hidden_states, params['gate'])
    routing_probs = jax.nn.softmax(router_logits, axis=-1)
    
    topk_probs, topk_indices = jax.lax.top_k(routing_probs, config.num_experts_per_tok)
    topk_probs = topk_probs / jnp.sum(topk_probs, axis=-1, keepdims=True)

    return topk_probs, topk_indices


### Expert MLP & MoE Execution

Each expert is a **SwiGLU** MLP — a gated variant that multiplies an element-wise gate (after SiLU) with the up-projection, then projects back down:

```
out = down_proj( SiLU(gate_proj(x)) ⊙ up_proj(x) )
```

`call_topk_experts` uses `vmap` to run the top-8 experts on a single token in parallel. `batch_moe_executor` iterates over all tokens with `jax.lax.scan` — a compile-friendly loop that gathers each token's expert parameters on-the-fly and sums their weighted outputs.

In [ ]:
@partial(jax.vmap, in_axes=(None, 0))
def call_topk_experts(
    token: jnp.ndarray, # (hidden_size,)
    expert_params: dict,
):
    gate_out = jax.nn.silu(jnp.dot(token, expert_params['gate_proj'])) * jnp.dot(token, expert_params['up_proj'])
    return jnp.dot(gate_out, expert_params['down_proj'])

def batch_moe_executor(
    tokens: jnp.ndarray,    # (seq_len, hidden_size)
    probs: jnp.ndarray,     # (seq_len, num_experts_per_tok)
    indices: jnp.ndarray,   # (seq_len, num_experts_per_tok)
    all_stacked_params: dict,
):
    def one_token_step(
        carry: None,
        x: tuple,
    ):
        token, prob, index = x
        topk_params = jax.tree_util.tree_map(lambda x: x[index], all_stacked_params)
        expert_outputs = call_topk_experts(token, topk_params)
        token_out = jnp.sum(expert_outputs * prob[:, None], axis=0)
        return None, token_out
        
    _, hidden_states = jax.lax.scan(one_token_step, None, (tokens, probs, indices))
    
    return hidden_states


### Decoder Block

One full transformer layer — the unit repeated 48 times. Both sub-layers use **pre-normalisation** (norm before the operation, not after):

```
x → RMSNorm → GQA → residual add
x → RMSNorm → MoE Router → top-8 experts → weighted sum → residual add
```

In [ ]:
def qwen3_decoder(
    hidden_states: jnp.ndarray, # (seq_len, hidden_size)
    params: dict,
    config: Qwen3MoEConfig,
    position_embeddings: tuple,
    start_pos: int,
    layer_cache: dict,
):
    residual = hidden_states
    hidden_states = rms_norm(hidden_states, params['input_norm']['scale'])
    hidden_states, layer_cache = qwen3_attention(hidden_states, params, config, position_embeddings, start_pos, layer_cache)
    hidden_states = residual + hidden_states
    
    residual = hidden_states
    hidden_states = rms_norm(hidden_states, params['post_attn_norm']['scale'])
    
    topk_probs, topk_indices = qwen3_router(hidden_states, params['mlp'], config)
    stacked_experts = params['mlp']['experts']
    hidden_states = batch_moe_executor(hidden_states, topk_probs, topk_indices, stacked_experts)
    
    hidden_states = residual + hidden_states

    return hidden_states, layer_cache


### Prefill Pass

Processes the entire prompt in a **single forward pass** — all tokens in parallel. `donate_argnames=('cache',)` hands the old cache buffer back to XLA so it can be reused in-place, avoiding a redundant allocation. `start_pos` is `static_argnames` so XLA specialises the graph for position 0.

In [ ]:
from functools import partial

@partial(
    jax.jit,
    static_argnames=('config', 'start_pos'),
    donate_argnames=('cache',)
)
def qwen_model_prefill(
    input_ids: jnp.ndarray, # (seq_len,)
    m: dict,
    cache: list,
    rotary_embeddings_table: tuple,
    config: Qwen3MoEConfig,
    start_pos: int = 0,
):
    input_ids = jax.device_put(input_ids, replicate_1d_sharding)
    input_embeddings = m['embed'][input_ids]
    
    hidden_states = input_embeddings

    seq_len = hidden_states.shape[0]
    position_ids = jnp.arange(seq_len) + start_pos
    position_embeddings = rope_lookup(position_ids, rotary_embeddings_table)
    
    updated_cache = []
    for i in range(config.num_hidden_layers):
        hidden_states, layer_cache = qwen3_decoder(hidden_states, m['layers'][i], config, position_embeddings, start_pos, cache[i])
        updated_cache.append(layer_cache)

    hidden_states = rms_norm(hidden_states, m['final_norm'])
    logits = jnp.dot(hidden_states, m['lm_head'])
    
    return logits, updated_cache


### Single-Step Generation Pass

The autoregressive decode step — processes exactly **one new token** at a time, reading from the KV cache filled during prefill. Almost identical to `qwen_model_prefill`, but `start_pos` is a regular (dynamic) argument here since it changes every step.

In [ ]:
@partial(
    jax.jit,
    static_argnames=('config',),
    donate_argnames=('cache',)
)
def qwen_model_generate(
    input_ids: jnp.ndarray, # (1,)
    m: dict,
    cache: list,
    rotary_embeddings_table: tuple,
    config: Qwen3MoEConfig,
    start_pos: int,
):
    input_ids = jax.device_put(input_ids, replicate_1d_sharding)
    input_embeddings = m['embed'][input_ids]
    hidden_states = input_embeddings

    seq_len = hidden_states.shape[0]
    position_ids = jnp.arange(seq_len) + start_pos
    position_embeddings = rope_lookup(position_ids, rotary_embeddings_table)

    updated_cache = []
    for i in range(config.num_hidden_layers):
        hidden_states, layer_cache = qwen3_decoder(hidden_states, m['layers'][i], config, position_embeddings, start_pos, cache[i])
        updated_cache.append(layer_cache)

    hidden_states = rms_norm(hidden_states, m['final_norm'])
    logits = jnp.dot(hidden_states, m['lm_head'])

    return logits, updated_cache


### Token Sampling

Applies temperature scaling to the last-position logits and samples with `jax.random.categorical`. Lower temperature → sharper distribution → more deterministic output. Pass `key=None` to use a fixed seed.

In [ ]:
def sample_token(
    logits: jnp.ndarray,    # (vocab_size,)
    temperature: float = 1.0,
    key: Optional[jax.Array] = None,
):
    if key is None:
        key = jax.random.PRNGKey(0)
        
    scaled_logits = logits / temperature
    probs = jax.nn.softmax(scaled_logits)
    token = jax.random.categorical(key, jnp.log(probs))
    
    return token


### Tokenise & Pad Input

Tokenise the prompt and pad to `max_seq_len` with zeros. We track `actual_len` so we can index `logits[actual_len - 1]` at the end of prefill — the logit for the last real token, which predicts the first new token.

In [ ]:
MAX_PROMPT_LEN = config.max_seq_len
tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_PATH)

def prepare_input(
    prompt: str,
):
    tokens = tokenizer(prompt, return_tensors="np")['input_ids'][0]
    actual_len = len(tokens)
    
    padded_tokens = jnp.pad(tokens, (0, MAX_PROMPT_LEN - actual_len))
    return jnp.array(padded_tokens), actual_len


### Choose a Prompt

Set your prompt here. `prepare_input` tokenises it and pads to `max_seq_len` — `actual_len` records how many real tokens are in the sequence.

In [ ]:
prompt = "The best country is"
input_ids, actual_len = prepare_input(prompt)
start_pos = 0

### VRAM Check

A quick sanity check on how much GPU memory the model weights occupy. Note this excludes the KV cache and activation memory needed during the forward pass.

In [ ]:
params_size = sum(x.nbytes for x in jax.tree_util.tree_leaves(m))
print(f"Weights in VRAM: {params_size / 1e9:.2f} GB")

### Run Prefill

Allocate the KV cache, run the prefill pass on the padded prompt, and sample the first new token from `logits[actual_len - 1]`. The first call compiles the function — expect a wait.

In [ ]:
cache = [
    get_kv_cache((config.max_seq_len, config.num_key_value_heads, config.head_dim), sharding=kv_cache_sharding) 
    for _ in range(config.num_hidden_layers)
]

key = jax.random.PRNGKey(42)

print("Generating: ", end="", flush=True)
logits, updated_cache = qwen_model_prefill(input_ids, m, cache, rotary_embeddings_table, config, start_pos)
next_token = sample_token(logits[actual_len - 1], temperature=0.2, key=key)

all_tokens = list(input_ids[:actual_len]) + [int(next_token)]
print(prompt, end="", flush=True)
print(tokenizer.decode([int(next_token)]), end="", flush=True)

### Greedy Decode Loop (Python)

A simple Python loop for quick testing — runs 30 autoregressive steps, calling `qwen_model_generate` one token at a time. Each call is separately JIT-compiled, so expect a pause on the first step. Useful for debugging but not optimal for throughput.

In [ ]:
cache = updated_cache

for step in range(30):
    start_pos = actual_len + step
    
    logits, cache = qwen_model_generate(jnp.array([next_token]), m, cache, rotary_embeddings_table, config, start_pos)
    next_token = sample_token(logits[-1], temperature=0.3, key=key)
    all_tokens.append(int(next_token))
    
    print(tokenizer.decode([int(next_token)]), end="", flush=True)

print("\n\nFull output:", tokenizer.decode(all_tokens))

### Compiled Generation with `jax.lax.scan`

Instead of a Python loop, we fold the entire generation loop into the XLA computation graph using `jax.lax.scan`. This compiles all `num_tokens` steps as a **single operation** — eliminating Python overhead and enabling better hardware utilisation.

`num_tokens` is a `static_argnames` so XLA knows the loop count at compile time and can fully unroll or optimise it. The trade-off: changing `num_tokens` triggers a recompile.

In [ ]:
@partial(jax.jit, static_argnames=('num_tokens',))
def compiled_generate(
    token: jnp.ndarray,     # (1,)
    m: dict,
    cache: list,
    key: jax.Array,
    start_pos: jnp.ndarray, # ()
    num_tokens: int,
):
    initial_carry = (token, cache, key, start_pos)
    
    def generate_step(
        carry: tuple,
        _,
    ):
        token, current_cache, rng_key, start_pos = carry
        
        logits, next_cache = qwen_model_generate(
            token, m, current_cache, rotary_embeddings_table, config, start_pos
        )
        
        rng_key, subkey = jax.random.split(rng_key)
        next_token = sample_token(logits[-1], temperature=0.6, key=subkey)
        
        new_carry = (
            jnp.array([next_token]),
            next_cache,
            rng_key,
            start_pos + 1
        )
        return new_carry, next_token
    
    final_carry, all_tokens = jax.lax.scan(
        generate_step, initial_carry, None, length=num_tokens
    )
    return final_carry, all_tokens


### Run Compiled Generation

The first call triggers compilation of the full `num_tokens`-step scan — expect a wait proportional to loop depth. Once compiled, re-running with the same `num_tokens` is fast.

In [ ]:
initial_start_pos = actual_len  

final_carry, tokens = compiled_generate(
    jnp.array([next_token]),
    m,
    updated_cache,
    key,
    jnp.array(actual_len),
    num_tokens=200
)

tokens_list = tokens.tolist()
print("Full output:", tokenizer.decode(tokens_list))